In [2]:
import pandas as pd
import numpy as np
import cvxpy as cp
from hmmlearn.hmm import GaussianHMM
from tqdm import tqdm
import os
import warnings

# Silenciar avisos
warnings.filterwarnings('ignore')

# ==============================================================================
# MÓDULO 1: O MOTOR DE OTIMIZAÇÃO (HMM + EWMA + L1)
# ==============================================================================
def otimizacao_condicional_hmm_ewma(retornos_historicos, pesos_atuais, custo_transacional=0.02, aversao_risco=10.0):
    
    # PASSO 1: DETECÇÃO DO REGIME DE MERCADO VIA HMM
    retornos_mercado = retornos_historicos.mean(axis=1).values.reshape(-1, 1)
    
    modelo_hmm = GaussianHMM(n_components=2, covariance_type="full", random_state=42)
    modelo_hmm.fit(retornos_mercado)
    estado_atual = modelo_hmm.predict(retornos_mercado)[-1]
    
    # PASSO 2: CÁLCULO EWMA MODULADO PELO REGIME
    alpha_dinamico = 0.04 if estado_atual == 0 else 0.10
        
    ewma_dados = retornos_historicos.ewm(alpha=alpha_dinamico)
    vetor_retornos_esp = ewma_dados.mean().iloc[-1].values
    
    matriz_cov_ewma = ewma_dados.cov().iloc[-len(retornos_historicos.columns):].values
    
    # PASSO 3: OTIMIZAÇÃO CONVEXA COM PENALIDADE L1
    n_ativos = len(retornos_historicos.columns)
    w = cp.Variable(n_ativos)
    
    matriz_cov_ewma = matriz_cov_ewma + np.eye(n_ativos) * 1e-9
    
    risco = cp.quad_form(w, cp.psd_wrap(matriz_cov_ewma))
    retorno = vetor_retornos_esp.T @ w
    
    giro_l1 = cp.norm(w - pesos_atuais, 1)
    custo_total = custo_transacional * giro_l1
    
    #funcao_objetivo = cp.Maximize(retorno - (aversao_risco * risco) - custo_total)
    funcao_objetivo = cp.Minimize(risco + custo_total)
    restricoes = [cp.sum(w) == 1, w >= 0]
    
    problema = cp.Problem(funcao_objetivo, restricoes)
    problema.solve(solver=cp.ECOS) 
    
    if w.value is None:
        raise ValueError("Otimizador não encontrou solução convexa.")
        
    return w.value

# ==============================================================================
# MÓDULO 2: O LAÇO DO BACKTEST (JANELAS MÓVEIS)
# ==============================================================================
def executar_backtest_dinamico(diretorio_dados, custo_transacional_pct=0.02, aversao_risco=10.0, meses_treino_inicial=60):
    print("=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===")
    
    # 1. Carregar Retornos
    caminho_retornos = os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv")
    print("1. A carregar a matriz de retornos contínua...")
    df_retornos = pd.read_csv(caminho_retornos, index_col='Data', parse_dates=True)
    
    colunas_ativos = [col for col in df_retornos.columns if col != 'IBOV']
    df_ativos = df_retornos[colunas_ativos]
    n_ativos = len(colunas_ativos)
    
    # 2. Carregar CDI e SELIC Externos
    print("2. A carregar taxas CDI e SELIC de fontes externas...")
    caminho_cdi = os.path.join(diretorio_dados, "CDI_2010_2026.xlsx")
    caminho_selic = os.path.join(diretorio_dados, "SELIC_2010_2026.xlsx")
    
    df_cdi = pd.read_excel(caminho_cdi)
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    df_selic = pd.read_excel(caminho_selic)
    df_selic = df_selic.rename(columns={'Date': 'Data', 'SELIC': 'SELIC'}).set_index('Data')
    df_selic.index = pd.to_datetime(df_selic.index)
    
    serie_cdi = df_cdi['CDI'].reindex(df_ativos.index).ffill()
    
    # Excesso de Retorno (Prémio de Risco) para passar ao HMM
    df_excesso = df_ativos.sub(serie_cdi, axis=0)
    
    print("3. A configurar pontos de rebalanceamento mensais...")
    datas_rebalanceamento = df_excesso.groupby([df_excesso.index.year, df_excesso.index.month]).apply(lambda x: x.index[-1]).values
    
    indice_inicio_backtest = meses_treino_inicial
    datas_backtest = datas_rebalanceamento[indice_inicio_backtest:]
    
    pesos_atuais = np.array([1.0 / n_ativos] * n_ativos)
    
    historico_pesos = pd.DataFrame(index=datas_backtest, columns=colunas_ativos, dtype=float)
    historico_turnover = pd.Series(index=datas_backtest, dtype=float)
    
    print(f"4. A iniciar a máquina do tempo: {len(datas_backtest)} rebalanceamentos a processar...")
    
    for data_corte in tqdm(datas_backtest):
        # O HMM e o Otimizador analisam o Excesso de Retorno
        dados_janela = df_excesso.loc[:data_corte]
        
        try:
            novos_pesos = otimizacao_condicional_hmm_ewma(
                retornos_historicos=dados_janela,
                pesos_atuais=pesos_atuais,
                custo_transacional=custo_transacional_pct,
                aversao_risco=aversao_risco
            )
            
            novos_pesos = np.where(novos_pesos < 1e-4, 0, novos_pesos)
            novos_pesos /= np.sum(novos_pesos) 
            
        except Exception as e:
            data_formatada = pd.to_datetime(data_corte).date()
            print(f"\n   -> Aviso na data {data_formatada}: Solução não convergiu. Mantendo pesos. ({e})")
            novos_pesos = pesos_atuais.copy()
            
        turnover_mensal = np.sum(np.abs(novos_pesos - pesos_atuais)) / 2.0
        
        historico_pesos.loc[data_corte] = novos_pesos
        historico_turnover.loc[data_corte] = turnover_mensal
        
        pesos_atuais = novos_pesos.copy()

    print("\n5. A finalizar o Backtest e a exportar as alocações dinâmicas...")
    caminho_historico = os.path.join(diretorio_dados, "historico_alocacao_HMM_EWMA_minima_variancia.csv")
    caminho_turnover = os.path.join(diretorio_dados, "historico_turnover_minima_variancia.csv")
    
    historico_pesos.to_csv(caminho_historico)
    historico_turnover.to_csv(caminho_turnover)
    
    print("=== RESUMO DA SIMULAÇÃO ===")
    print(f"Total de meses rebalanceados: {len(historico_pesos)}")
    print(f"Turnover médio mensal da carteira: {historico_turnover.mean() * 100:.2f}%")
    print("===========================================================")
    
    print("6. A formatar o Painel de Evolução por Ação (Long Data)...")
    df_painel = historico_pesos.reset_index().melt(
        id_vars='index', 
        var_name='Ativo', 
        value_name='Peso'
    )
    df_painel.rename(columns={'index': 'Data'}, inplace=True)
    df_painel = df_painel[df_painel['Peso'] > 0.0001].sort_values(by=['Data', 'Peso'], ascending=[True, False])
    
    caminho_painel = os.path.join(diretorio_dados, "painel_alocacao_mensal_minima_variancia.csv")
    df_painel.to_csv(caminho_painel, index=False)
    print(f"   -> Painel de visualização salvo em: {caminho_painel}")

    return historico_pesos, historico_turnover

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"

pesos_no_tempo, giro_carteira = executar_backtest_dinamico(
    diretorio_dados=pasta_dados, 
    custo_transacional_pct=0.002, 
    aversao_risco=10.0, 
    meses_treino_inicial=60
)

=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===
1. A carregar a matriz de retornos contínua...
2. A carregar taxas CDI e SELIC de fontes externas...
3. A configurar pontos de rebalanceamento mensais...
4. A iniciar a máquina do tempo: 132 rebalanceamentos a processar...


100%|██████████| 132/132 [06:35<00:00,  3.00s/it]


5. A finalizar o Backtest e a exportar as alocações dinâmicas...
=== RESUMO DA SIMULAÇÃO ===
Total de meses rebalanceados: 132
Turnover médio mensal da carteira: 0.31%
6. A formatar o Painel de Evolução por Ação (Long Data)...
   -> Painel de visualização salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\painel_alocacao_mensal_minima_variancia.csv


=== INÍCIO DO BACKTEST: JANELAS MÓVEIS (HMM + EWMA + L1) ===
1. A carregar a matriz de retornos contínua...
2. A configurar pontos de rebalanceamento mensais...
3. A iniciar a máquina do tempo: 132 rebalanceamentos a processar...
100%|██████████| 132/132 [07:09<00:00,  3.26s/it]

4. A finalizar o Backtest e a exportar as alocações dinâmicas...
=== RESUMO DA SIMULAÇÃO ===
Total de meses rebalanceados: 132
Turnover médio mensal da carteira: 0.51%
===========================================================
5. A formatar o Painel de Evolução por Ação (Long Data)...
   -> Painel de visualização salvo em: C:\VSCodeWorkspace\TCC_Escrito\data\painel_alocacao_mensal.csv